# Online training with TorchFort

## Case
In this exercise, we consider the prediction of wall-bounded turbulence from wall quantities using Deep Learning (DL), as presented in ([Guastoni et al. 2020](https://iopscience.iop.org/article/10.1088/1742-6596/1522/1/012022)). Specifically, we want to online/in-situ train a Convolutional Neural Network (CNN) model which takes the wall shear stress field $\boldsymbol{\tau}(x,y,z|_{z=0},t)$ at the solid wall as an input and predicts the velocity field $\mathbf{v}(x,y,z|_{z=h},t)$ at a given height $h$. Online training in this context means that the training and simulation processes are tightly coupled i.e. the inputs and labels are read directly from the running simulation without writing anything to disk.

Figure 1 illustrates the setup. Please note that in this case, the last dimension "$z$" corresponds to the wall-normal direction.

<img src="channel_flow.png" width="400"/>
Figure 1. Channel flow configuration.


## Simulation Code

The simulation code used in this exercise is CaNS (Canonical Navier-Stokes). CaNS is an open-source, massively-parallel numerical solver written in modern Fortran for simulating incompressible turbulent flows in canonical geometries. It uses a second-order finite-difference method on a structured Cartesian grid combined with a fast direct Poisson solver (via FFT-based decomposition), and supports GPU acceleration through CUDA Fortran/OpenACC for high-performance CFD on modern HPC systems.



## Exercise structure
At the ISC tutorial we only have access to a single GPU, but we will do the implementation such that it generalizes to a multi-GPU case. The exercise is structured as follows. 

First, we create the neural network model and run the simulation without any modifications from $t=0$ to $t=4000$ to avoid initial transients and collect necessary normalization statistics. Second, we add the online training capability. Third, we continue the simulation from $t=4000$ onwards with training switched on. Finally, we visualize and analyze the results.

## Online learning algorithm details
As the simulation advances in time, each process accumulates 32 local data samples of wall shear stress slices at the bottom wall to serve as inputs, and velocities at height $h$ to serve as labels. Both arrays will have dimensions of ($N_x$, $N_y$, $C$, $B$), where $N_i$ denotes the number of grid points in the spatial directions in a local partition, $C$ is the number of variables (and CNN input/output channels), and $B$ is the batch size. This is followed by `MPI_Alltoallv` after which each process will have $N_{GPU}$ number of local slices which can be reshaped as full slices ($\mathcal{N}_x$, $\mathcal{N}_y$, $C$, $B$), where $\mathcal{N}_i$ is now the global number of grid points in the domain. During the training step, each process will pick $B/N_{GPU}$ unique full slices and process them in data-parallel fashion. TorchFort abstracts the model gradient averaging in the `torchfort_train` method, so the effective $B$ remains 32. The coupled online simulation+training process will continue until we have reached a user specified epoch which is 200 batches (6400 samples) by default. This is followed by a validation epoch where we do not train but compare the inference and ground truth results. The default length of the validation epoch is 20 batches (640 samples). 

The presentation slides, TorchFort documentation [https://nvidia.github.io/TorchFort/](https://nvidia.github.io/TorchFort/), and this description should contain all necessary information to complete the exercise. The instructor and TAs are more than happy to help!

## Preparation steps
### Step 1 - Model creation
As a first step we need to consider model creation that corresponds to the given setup. In [./files/python_model/fcn.py](./files/python_model/fcn.py), we have implemented a simple CNN template using PyTorch. Fill in the correct number of input and output channels and add the TorchScript conversion and save function calls. Once you are ready, execute the next cell to run the Python code. The following cell will copy the saved model into the correct case folder.

In [ ]:
!python ./files/python_model/fcn.py

In [ ]:
!cp ./files/python_model/cans_fcn.pt ./files/reconstruction_case/

### Step 2 - Normalization stats
Data normalization with online training approach can be challenging as there is no fixed dataset on disk to compute necessary statistics. In this exercise, we wish to use standard Z-score normalization (zero mean, unit standard deviation) for both inputs and outputs, given by 

$\hat{x} = \frac{x - \langle x \rangle}{\sqrt{\langle (x - \langle x \rangle)^2 \rangle}}$,

where the chevrons denote time-averaging. Compile and run the code without any modifications to collect the statistics for normalisation by executing the following cells. Stats will be saved as .txt files into [./files/reconstruction_case/data/](./files/reconstruction_case/data/) folder. In addition, this run also saves a simulation checkpoint from which you can start the training phase to avoid any initial transients. This simulation phase takes you from $t=0$ to $t=4000$ time units.

In [ ]:
!bash -c "cd /CaNS && make libs && make"

In [ ]:
!cp ./files/reconstruction_case/input.stats.nml ./files/reconstruction_case/input.nml

In [ ]:
!bash -c "cd ./files/reconstruction_case && \
    mpirun -np 1 --allow-run-as-root --bind-to none /CaNS/run/cans"

## Enabling online training and inference.
The following exercises require you to modify the CaNS source code. All modifications should be done into [./files/CaNS_src_updates/main.f90](./files/CaNS_src_updates/main.f90). Each step will explicitly state which line to modify and in the source code they are annotated with TODO statements. 

### Step 3 - Imports
Include TorchFort library and CUDA Fortran (if OpenACC is used) to expose their APIs on lines marked with `TODO1` and `TODO2`.

Context: These lines are placed in a section that contains all module imports.

### Step 4 - Extra Arrays
TorchFort API accepts multi-dimensional Fortran arrays as inputs and the entire simulation state arrays could be directly passed as inputs and/or labels to functions like `torchfort_train`. However, in this exercise we are only interested in specific slices, wall shear stresses at the bottom wall (the inputs), and the flow velocities at the evaluation height $h$ (labels), so we should allocate arrays to hold these. 

Using the information in "Online Algorithm Details" section, implement the following.

- On line marked with `TODO3`, declare arrays for the local slices named `input_local`, `label_local`.
- On line marked with `TODO4`, declare arrays for the full slices named `input`, `output` and `label`.
- On line marked with `TODO5`, declare a variable named `loss_value` to hold the loss value whilst training.
- On line marked with `TODO6`, allocate memory for local input and label. (Hint: after all-to-all we will need memory for $B * N_{GPU}$ local slices)
- On line marked with `TODO7`, allocate memory for the full input, label and output.

You have access to the following variables `n(1:3)` (number of nodes in a partition in direction 1=x, 2=y, 3=z), `ng(1:3)` (number of global nodes in direction 1=x, 2=y, 3=z), `trainbs` (batch size) and `nranks` (number of MPI ranks).

Contex: These lines are placed in sections that handle variable declarations and memory allocations.

Hint: You can always search for a variable to see their type and other uses e.g. `integer , dimension(3) :: n`. Also mimicking the surrounding code's syntax is often enough, even if you are not familiar with Fortran.

### Step 5 - Model initialization
Implement the model creation API calls at `TODO8` (single GPU), `TODO9` (multi GPU), `TODO10` (single CPU), and `TODO11` (multi CPU) 

You have access to the following variables.

`model_name`, `model_config_file`, `MPI_COMM_WORLD`, `dev` (GPU device id), `TORCHFORT_DEVICE_CPU` (CPU device id).

Note: These lines placed in the section that handles solver related initialisations.

### Step 6 - Model restart
To support restarting from a checkpoint, implement the checkpoint loading API call on line marked with `TODO12`.

You have access to the following variables.

`torchfort_ckpt` (checkpoint file name), `isteptrain` (number of training steps), `istepval` (number of validation steps).

Note: This line is placed in the section that handles solver related initialisations.

### Step 7 - Training step
Implement the model training step API call on line marked with `TODO13`.

Context: This line is placed within the primary `do while` loop that advances the solution in time and is active only when certain conditionals are met i.e. after warm-up phase and not validating.

### Step 8 - Inference step
Implement the model inference step API calls on lines marked with `TODO14` and `TODO15`. The former corresponds to running the code in inference mode and the second corresponds to validation step occurring after epochs.

Context: These lines are placed within the primary `do while` loop that advances the solution in time and are only active when we are validating or performing test inference.

### Step 9 - Model save
Implement the model checkpoint save API call on line marked with `TODO16`.

You have access to the following variables.

`model_name` and `filename`.

Context: This line is placed within the primary time loop and activates only once we have completed validation epoch and want to save.


### Step 10 - Run the case
Once you are ready with all the steps, copy your source code changes to `main.f90` by executing the cell below. It simply overwrites the original `main.f90` file.

In [ ]:
!cp ./files/CaNS_src_updates/* ./CaNS/src/

Recompile the code by executing the following cell.

In [ ]:
!bash -c "cd /CaNS && make"

Take some time to explore the simulation and training config files. 

First, look into [./files/reconstruction_case/input.train.nml](./files/reconstruction_case/input.train.nml). There are many parameters but to highlight a few, we specify that the simulation is restarted from the previous simulation checkpoint at $t=4000$ since `restart = T` and we run the simulation until $t=6200$ since `time_max = 6200`. In the TorchFort section at the bottom, we see that we specify batch size of 32 since `trainbs = 32`, one training "epoch" is specified as 6400 since `nsamples_train = 6400` and the validation "epoch" is specified as `nsamples_val = 640`. This means that as we run the simulation, we train with 200 batches of data and then validate for 20 batches. Validation epochs will also output HDF5 files from which you can visualise the NN prediction and real result and save the model checkpoint to disk.

Second, look into [./files/reconstruction_case/config_jit.yaml](./files/reconstruction_case/config_jit.yaml) and fill in the TODO statements.

Hints: `file_name` should correspond to the model created in Step 1. What should be the Loss function? Adam optimizer with a learning rate 1e-3 has been found to work well.

Finally, start the coupled simulation + training phase by executing the next cells.

In [ ]:
!cp ./files/reconstruction_case/input.train.nml ./files/reconstruction_case/input.nml

In [ ]:
!bash -c "cd ./files/reconstruction_case && \
    mpirun -np 1 --allow-run-as-root --bind-to none /CaNS/run/cans"

## Plotting
Let us plot the validations results here. The results are stored in [./files/reconstruction_case/data](./files/reconstruction_case/data) 

In [ ]:
import h5py
import matplotlib.pyplot as plt

data = h5py.File(TODO, "r")
print(list(data))
print(data['label'].shape)

In [ ]:
fig = plt.figure(figsize=(7, 10))
for i, var in enumerate(['u', 'v', 'w']):
    plt.subplot(3, 2, i*2 + 1)
    plt.imshow(data["pred"][0,i,:,:])
    plt.colorbar()
    plt.title(var)

    plt.subplot(3, 2, i*2 + 2)
    plt.imshow(data["label"][0,i,:,:])
    plt.colorbar()
    plt.title(var)
    

## Analysis and Questions
What could you say about these results? \
What could you say about the training loss and validation losses? \
How could we improve the results? \
What are the advantages and disadvantages of online learning? \
Why do we need to gather all local slices to form a full slice for training although CNNs should be translation invariant? \
Could we do the multi-gpu domain parallelism differently?

## Additional Exercise

The previous training clearly was not long enough. Let's restart the training from the previous model checkpoint! You only need to modify the input.nml and rerun the code. You should observe that the training loss now starts from approximately at the same level as the last reported value.